# 03 — Baseline Model: Linear Regression

Trains a Linear Regression on the model-ready SFR data (from `02_preprocessing.ipynb`) and evaluates R² on the held-out test set. These results are the baseline all later models must beat.

**Target:** `log_ClosePrice` (log-transformed sale price)

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
TARGET = "log_ClosePrice"


## 1. Load model-ready data

In [2]:
train = pd.read_csv("sfr_train_model_ready.csv")
test = pd.read_csv("sfr_test_model_ready.csv")

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_test, y_test = test.drop(columns=[TARGET]), test[TARGET]

print(f"Train: {X_train.shape[0]:,} rows x {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]:,} rows x {X_test.shape[1]} features")
assert list(X_train.columns) == list(X_test.columns)

Train: 49,607 rows x 29 features
Test:  11,996 rows x 29 features


## 2. Train Linear Regression

In [3]:
lr = LinearRegression()
lr.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


## 3. Evaluate — R² on test set

In [4]:
pred_train = lr.predict(X_train)
pred_test = lr.predict(X_test)

r2_train = r2_score(y_train, pred_train)
r2_test = r2_score(y_test, pred_test)
rmse_log = np.sqrt(mean_squared_error(y_test, pred_test))

# Back-transform to dollars for interpretability
mae_dollars = mean_absolute_error(np.exp(y_test), np.exp(pred_test))

print(f"R2 (train):        {r2_train:.4f}")
print(f"R2 (test):         {r2_test:.4f}")
print(f"RMSE (test, log):  {rmse_log:.4f}")
print(f"MAE  (test, $):    ${mae_dollars:,.0f}")

R2 (train):        0.8659
R2 (test):         0.8643
RMSE (test, log):  0.2480
MAE  (test, $):    $262,260


## 4. Record baseline results

In [5]:
results = pd.DataFrame([{
    "model": "LinearRegression",
    "notebook": "03_baseline_model",
    "r2_train": round(r2_train, 4),
    "r2_test": round(r2_test, 4),
    "rmse_test_log": round(rmse_log, 4),
    "mae_test_dollars": round(mae_dollars, 0),
}])
results.to_csv("model_results.csv", index=False)
results

,model,notebook,r2_train,r2_test,rmse_test_log,mae_test_dollars
0,LinearRegression,03_baseline_model,0.8659,0.8643,0.248,262260.0


## 5. Top coefficients (sanity check)

Features are standardized, so coefficient magnitude ≈ relative importance.

In [6]:
coefs = pd.Series(lr.coef_, index=X_train.columns).sort_values(key=abs, ascending=False)
coefs.head(10).round(4)

PostalCode_te            0.2552
LivingArea               0.2227
Longitude               -0.1364
City_te                  0.1324
Latitude                -0.1311
BathroomsTotalInteger    0.1013
CountyOrParish_te        0.0673
HasHOA                  -0.0574
AssociationFee           0.0303
Levels_Two              -0.0281
dtype: float64

## Baseline results (recorded)

| Metric | Value |
|---|---|
| **R² (test)** | **0.8643** |
| R² (train) | 0.8659 |
| RMSE (test, log scale) | 0.2480 |
| MAE (test, dollars) | $262,260 |

Results saved to `model_results.csv`. Train and test R² are close, so the model is not overfitting — a linear fit simply can't capture more of the variance. Later models (tree ensembles, etc.) must beat **test R² = 0.8643**.